In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 2.7 Small Oscillations from a General Lagrangian

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume II — Analytical Mechanics",
    number="2.7",
    title="Small Oscillations from a General Lagrangian",
    blurb="Normal modes of any system near equilibrium: linearise the "
    "Lagrangian, then simultaneously diagonalise the mass and "
    "stiffness matrices — the generalised eigenproblem that underlies "
    "molecular vibrations, phonons, and coupled fields.",
    difficulty="advanced",
    estimate="80–100 min",
)

## Notebook overview

Almost any system, left near a stable equilibrium and nudged gently, rings. A
molecule, a bridge, a crystal lattice, a pair of coupled pendula: push them a
little and they oscillate, and the motion we see is always a superposition of
a few special patterns, the **normal modes**, each vibrating at its own fixed
frequency. This notebook builds the general machine that finds those modes for
*any* Lagrangian system, by linearising the dynamics about equilibrium and
solving a single matrix eigenproblem.

This is the culmination of the course's small-oscillations work, and it ties
three earlier threads together.
In [§1.5](../01-elementary-mechanics/coupled-oscillators.ipynb) we found the normal modes of a spring chain, but there the
masses were uncoupled, so the mass matrix was a multiple of the identity and an
*ordinary* eigenproblem sufficed. Here the kinetic energy couples the
coordinates (the mass matrix is genuinely non-diagonal), and the right tool is
the **generalised** eigenproblem $K\mathbf v = \omega^2 M\mathbf v$. In
[§1.3](../01-elementary-mechanics/double-pendulum.ipynb) we measured the two natural frequencies of the double pendulum by
taking an FFT of a chaotic-looking small-amplitude trajectory; here we derive
those same frequencies *exactly*, in closed form, as eigenvalues. And the
symbolic Euler–Lagrange engine from [§2.1](lagrangian-sympy.ipynb) does the linearisation for us,
straight from the Lagrangian.

> **How to read the checks.** Each exercise ends with a validation that
> compares our result to an expected physical fact. A ✗ does *not* by itself
> mean the answer is wrong: it means the output didn't match what the check
> expected, which may be a real error, a different-but-valid convention (a
> sign, an eigenvector's overall sign, an array order), or simply too tight a
> tolerance. Treat a ✗ as a prompt to locate the discrepancy; passing is
> strong evidence of correctness, not proof.

> **Scope.** This is a working review, not a textbook chapter. For the full
> theory of small oscillations (the normal-coordinate transformation, the
> simultaneous diagonalisation theorem, degeneracies from symmetry), see
> Nolting, *Theoretical Physics 2* {cite}`nolting2`, Goldstein, Poole & Safko,
> *Classical Mechanics* {cite}`goldstein` ch. 6, and Landau & Lifshitz,
> *Mechanics* {cite}`landau_mechanics` §V.

## Theory in brief

### Linearising a Lagrangian about equilibrium

Take a system with generalised coordinates $q_1,\dots,q_n$ and a natural
Lagrangian $\mathcal L = T - V$, where the kinetic energy is quadratic in the
velocities. Let $\mathbf q_0$ be a **stable equilibrium**: a point where the
generalised forces vanish, $\partial V/\partial q_i|_{\mathbf q_0} = 0$, and $V$
is a local minimum. Write small displacements $\eta_i = q_i - q_{0,i}$ and
expand to quadratic order.

The kinetic energy becomes a quadratic form in the velocities with the
(generally non-diagonal, symmetric, positive-definite) **mass matrix**

```{math}
:label: eq-massmat
T = \tfrac12\,\dot{\boldsymbol\eta}^{\mathsf T} M\, \dot{\boldsymbol\eta},
\qquad
M_{ij} = \frac{\partial^2 T}{\partial \dot q_i\,\partial \dot q_j}\bigg|_{\mathbf q_0},
```

and the potential, expanded about the minimum (the constant is irrelevant and
the linear term vanishes at equilibrium), becomes a quadratic form with the
symmetric **stiffness matrix**

```{math}
:label: eq-stiffmat
V = \tfrac12\,\boldsymbol\eta^{\mathsf T} K\, \boldsymbol\eta,
\qquad
K_{ij} = \frac{\partial^2 V}{\partial q_i\,\partial q_j}\bigg|_{\mathbf q_0}.
```

Stability means $V$ is a genuine minimum, i.e. $K$ is **positive definite**.

### The equations of motion and the generalised eigenproblem

Feeding the quadratic $\mathcal L = T - V$ into the Euler–Lagrange equations
{eq}`eq-el` gives a linear, coupled system,

```{math}
:label: eq-modes-eom
M\,\ddot{\boldsymbol\eta} = -K\,\boldsymbol\eta.
```

Look for synchronous solutions $\boldsymbol\eta(t) = \mathbf v\, e^{i\omega t}$,
in which every coordinate oscillates with the *same* frequency $\omega$ and
fixed relative amplitudes $\mathbf v$. Substituting turns {eq}`eq-modes-eom`
into the **generalised eigenvalue problem**

```{math}
:label: eq-geneig
K\,\mathbf v = \omega^2\, M\,\mathbf v.
```

The eigenvalues $\omega_k^2$ are the squared normal-mode frequencies; the
eigenvectors $\mathbf v_k$ are the **mode shapes**. Because $M \neq I$ in
general, this is *not* the ordinary eigenproblem $K\mathbf v = \lambda\mathbf v$
of [§1.5](../01-elementary-mechanics/coupled-oscillators.ipynb): the mass matrix sits on the right-hand side, weighting the
eigenproblem. That is the whole generalisation.

### Orthogonality and simultaneous diagonalisation

The mode shapes are not orthogonal in the usual sense, but **orthogonal with
respect to the mass matrix**:

```{math}
:label: eq-mortho
\mathbf v_i^{\mathsf T} M\, \mathbf v_j = \delta_{ij},
\qquad
\mathbf v_i^{\mathsf T} K\, \mathbf v_j = \omega_i^2\,\delta_{ij},
```

where we have normalised each mode so that $\mathbf v_i^{\mathsf T} M\mathbf
v_i = 1$. Collecting the eigenvectors as the columns of a **modal matrix**
$V = [\mathbf v_1\,\cdots\,\mathbf v_n]$, equation {eq}`eq-mortho` says
$V^{\mathsf T} M V = I$ and $V^{\mathsf T} K V = \operatorname{diag}(\omega_k^2)$:
the single transformation $V$ diagonalises **both** $M$ and $K$ at once. In the
**normal coordinates** $\boldsymbol\xi = V^{\mathsf T} M\,\boldsymbol\eta$ the
system decouples completely into $n$ independent oscillators
$\ddot\xi_k = -\omega_k^2\,\xi_k$: that is the point of the whole construction.

### Stability from the spectrum

Because $M$ is positive definite, the signs of the $\omega_k^2$ follow the
eigenvalues of $K$. If $K$ is positive definite every $\omega_k^2 > 0$ and the
system oscillates. If the equilibrium is a saddle, $K$ has a negative
eigenvalue, some $\omega_k^2 < 0$, and that mode's $e^{i\omega t}$ becomes a
real exponential $e^{|\omega| t}$: an **instability**, not an oscillation. The
spectrum diagnoses stability directly.

---
## Setup

We use SymPy to linearise a Lagrangian symbolically and `scipy.linalg.eigh` to
solve the generalised eigenproblem {eq}`eq-geneig`. The two helpers below are
the reusable core:

- `euler_lagrange(L, coords, t)`: the symbolic engine of [§2.1](lagrangian-sympy.ipynb), imported
  from `ecp.mechanics`, used to derive the exact (nonlinear) equations of motion.
- `linearize(L, coords, equilibrium)`: forms the mass and stiffness matrices
  {eq}`eq-massmat`, {eq}`eq-stiffmat` as Hessians of $\mathcal L$ evaluated at
  equilibrium: $M_{ij} = \partial^2\mathcal L/\partial\dot q_i\partial\dot q_j$
  and $K_{ij} = -\partial^2\mathcal L/\partial q_i\partial q_j$ (with the
  velocities set to zero, where $\mathcal L = -V$ for a natural system).

In [ ]:
import numpy as np
import sympy as sp
from scipy.integrate import solve_ivp
from scipy.linalg import eigh
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

from ecp import draw, validate
from ecp.mechanics import euler_lagrange
from ecp.animate import show

t = sp.symbols("t")  # the time variable shared by all dynamical coordinates


def linearize(L, coords, equilibrium):
    """Mass and stiffness matrices of a Lagrangian about an equilibrium.

    The second-order expansion gives M (from the kinetic energy) and K (from
    the potential) — eq-massmat and eq-stiffmat in the theory section — the
    input to the normal-mode eigenproblem.

    Parameters
    ----------
    L : sympy.Expr
        The Lagrangian.
    coords : sequence
        Coordinates.
    equilibrium : sequence
        Equilibrium coordinate values.

    Returns
    -------
    M, K : sympy.Matrix
        The mass and stiffness matrices.
    """
    n = len(coords)
    qdots = [sp.diff(q, t) for q in coords]
    at_eq = {qd: 0 for qd in qdots}
    at_eq.update({coords[i]: equilibrium[i] for i in range(n)})
    M = sp.zeros(n, n)
    K = sp.zeros(n, n)
    for i in range(n):
        for j in range(n):
            M[i, j] = sp.simplify(sp.diff(L, qdots[i], qdots[j]).subs(at_eq))
            K[i, j] = sp.simplify(-sp.diff(L, coords[i], coords[j]).subs(at_eq))
    return M, K


def double_pendulum_lagrangian():
    """The exact equal-mass, equal-length double-pendulum Lagrangian.

    Built symbolically in (θ1, θ2), the test system for the small-oscillation
    machinery.

    Returns
    -------
    tuple
        The Lagrangian and its coordinate symbols.
    """
    m, l, g = sp.symbols("m l g", positive=True)
    th1 = sp.Function("theta1")(t)
    th2 = sp.Function("theta2")(t)
    x1, y1 = l * sp.sin(th1), -l * sp.cos(th1)
    x2, y2 = x1 + l * sp.sin(th2), y1 - l * sp.cos(th2)
    T = sp.Rational(1, 2) * m * (sp.diff(x1, t) ** 2 + sp.diff(y1, t) ** 2)
    T += sp.Rational(1, 2) * m * (sp.diff(x2, t) ** 2 + sp.diff(y2, t) ** 2)
    V = m * g * y1 + m * g * y2
    return T - V, [th1, th2], (m, l, g)


# Numerical parameter values reused throughout (equal masses and lengths).
M_VAL, L_VAL, G_VAL = 1.0, 1.0, 9.81


def to_float_matrices(M_sym, K_sym, syms, vals):
    """Substitute numeric parameters and return float NumPy matrices.

    Parameters
    ----------
    M_sym, K_sym : sympy.Matrix
        Symbolic mass and stiffness matrices.
    syms : sequence
        Parameter symbols.
    vals : sequence
        Their numeric values.

    Returns
    -------
    M, K : numpy.ndarray
        The numeric matrices.
    """
    sub = dict(zip(syms, vals))
    M = np.array(M_sym.subs(sub)).astype(float)
    K = np.array(K_sym.subs(sub)).astype(float)
    return M, K


def bob_xy(th1, th2, L=L_VAL):
    """Cartesian positions of the two double-pendulum bobs from the angle series."""
    x1, y1 = L * np.sin(th1), -L * np.cos(th1)
    x2, y2 = x1 + L * np.sin(th2), y1 - L * np.cos(th2)
    return x1, y1, x2, y2

## Exercise 1 — From a Lagrangian to $M$ and $K$ (the linearisation engine)

Everything rests on extracting the two matrices {eq}`eq-massmat` and
{eq}`eq-stiffmat` from a Lagrangian. The mass matrix is the Hessian of
$\mathcal L$ in the velocities; the stiffness matrix is minus its Hessian in the
coordinates, both evaluated at equilibrium with the velocities set to zero. The
`linearize` helper in Setup does exactly this. The natural test system is the
small-angle double pendulum of [§1.3](../01-elementary-mechanics/double-pendulum.ipynb): its kinetic energy *couples*
$\dot\theta_1$ and $\dot\theta_2$, so the mass matrix had better come out
non-diagonal.

With equal masses $m$ and lengths $\ell$, the exact Lagrangian is built in
Setup. Linearising about the hanging equilibrium $\boldsymbol\theta_0 =
(0, 0)$ should give

$$ M = m\ell^2\begin{pmatrix} 2 & 1 \\ 1 & 1\end{pmatrix}, \qquad
   K = mg\ell\begin{pmatrix} 2 & 0 \\ 0 & 1\end{pmatrix}, $$

the off-diagonal of $M$ being the kinematic coupling of the two angles.

1. Build the exact double-pendulum Lagrangian and run `euler_lagrange` to
   see the full (nonlinear) accelerations it implies via {eq}`eq-el`.
2. Call `linearize` about $\boldsymbol\theta_0 = (0,0)$ to obtain $M$ and
   $K$ from {eq}`eq-massmat`–{eq}`eq-stiffmat`, and confirm the mass matrix
   is non-diagonal and both match the hand-derived matrices above
   (`sympy.simplify` on the differences).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    M_dp[0, 1] != 0,
    "the double-pendulum mass matrix is non-diagonal (the coordinates are coupled)",
    f"M[0,1] = {M_dp[0, 1]}",
)
validate.check(
    sp.simplify(M_dp - M_hand) == sp.zeros(2, 2)
    and sp.simplify(K_dp - K_hand) == sp.zeros(2, 2),
    "linearise reproduces the hand-derived small-angle M and K",
)

## Exercise 2 — The generalised eigenproblem → normal frequencies

With $M$ and $K$ in hand, the normal-mode frequencies are the eigenvalues of the
generalised problem {eq}`eq-geneig`, $K\mathbf v = \omega^2 M\mathbf v$. Crucially,
because $M$ is not the identity we must solve the *generalised* problem (passing
both $K$ and $M$ to `scipy.linalg.eigh`), not the ordinary eigenproblem of
[§1.5](../01-elementary-mechanics/coupled-oscillators.ipynb). For the equal double pendulum the characteristic equation factorises
in closed form to

$$ \omega_\pm^2 = (2 \pm \sqrt 2)\,\frac{g}{\ell}, $$

the two frequencies whose beating we extracted by FFT from a small-amplitude
trajectory back in [§1.3](../01-elementary-mechanics/double-pendulum.ipynb): now obtained exactly as eigenvalues.

**Your task.** Substitute numbers into $M$ and $K$, solve {eq}`eq-geneig` with
`eigh(K, M)`, and confirm the eigenfrequencies $\omega_k = \sqrt{\omega_k^2}$
equal $\sqrt{(2 \mp \sqrt 2)\,g/\ell}$.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    np.sort(omega),
    np.sort(omega_exact),
    "double-pendulum normal frequencies are √((2∓√2) g/ℓ) (the ω± of §1.3)",
    rtol=1e-9,
)

## Exercise 3 — $M$-orthogonality and simultaneous diagonalisation

The mode shapes are orthogonal *through the mass matrix*, {eq}`eq-mortho`: with
the modal matrix $V$ whose columns are the eigenvectors, $V^{\mathsf T} M V = I$
and $V^{\mathsf T} K V = \operatorname{diag}(\omega_k^2)$. The single
transformation $V$ therefore diagonalises **both** matrices simultaneously:
which is exactly why the normal coordinates $\boldsymbol\xi = V^{\mathsf T} M
\boldsymbol\eta$ decouple the dynamics. `eigh(K, M)` already returns
$M$-orthonormal eigenvectors, so this is a direct check on the structure.

**Your task.** Form the matrix products $V^{\mathsf T} M V$ and
$V^{\mathsf T} K V$ and confirm the first is the identity and the second is
diagonal with the $\omega_k^2$ on its diagonal.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(VtMV, np.eye(2), "the modes are M-orthonormal (VᵀMV = I)", atol=1e-8)
validate.close(
    offdiag,
    np.zeros((2, 2)),
    "the modes simultaneously diagonalise K (VᵀKV is diagonal)",
    atol=1e-8,
)

## Exercise 4 — Mode shapes: in-phase and out-of-phase

Each eigenvector $\mathbf v_k$ of {eq}`eq-geneig` is a **mode shape**: the fixed
pattern of relative amplitudes the coordinates hold while oscillating at
$\omega_k$. For the double pendulum the two shapes are physically vivid. The
**low-frequency mode** ($\omega_-$) has the two bobs swinging the *same* way
(in phase), so the system is soft and slow; the **high-frequency mode**
($\omega_+$) has them swinging *oppositely* (out of phase), a stiffer, faster
motion. (Only the *ratio* of the components is physical: an eigenvector's
overall sign is arbitrary.)

**Your task.** Read the two columns of the eigenvector matrix $V$ returned
by `scipy.linalg.eigh`, and confirm the low mode has components of the *same*
sign (in phase) while the high mode has components of *opposite* sign (out of
phase).

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    (v_low[0] * v_low[1] > 0) and (v_high[0] * v_high[1] < 0),
    "mode shapes match the expected patterns: low mode in phase, high mode out of phase",
    f"low product {v_low[0] * v_low[1]:+.3f}, high product {v_high[0] * v_high[1]:+.3f}",
)

## Exercise 5 — A pure mode oscillates at one frequency (worked animation)

The defining property of a normal mode: start the system *exactly* in one
eigenvector and it stays there, every coordinate tracing a sinusoid at the
single frequency $\omega_k$, the shape frozen. We saw this for the uncoupled
spring chain in [§1.5](../01-elementary-mechanics/coupled-oscillators.ipynb); here it holds for a genuinely coupled-mass system,
and it is the physical content of the simultaneous diagonalisation {eq}`eq-mortho`.

We integrate the linearised equations {eq}`eq-modes-eom`, $\ddot{\boldsymbol\eta}
= -M^{-1}K\,\boldsymbol\eta$, with `scipy.integrate.solve_ivp` (`DOP853`) on a dense
`t_eval` from the initial condition $\boldsymbol\eta(0) = A\,\mathbf v_-$ (a pure
low mode) at rest, and animate the resulting
small-amplitude double pendulum. Two independent checks confirm the physics of
the integrated data: the energy $E = \tfrac12\dot{\boldsymbol\eta}^{\mathsf T} M
\dot{\boldsymbol\eta} + \tfrac12\boldsymbol\eta^{\mathsf T} K\boldsymbol\eta$ is
conserved, and there is **no leakage**: the projection onto the *other* mode,
$\xi_{+}(t) = \mathbf v_+^{\mathsf T} M\,\boldsymbol\eta(t)$, stays zero.

This is the worked animation; you build the second in Exercise 6.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.conserved(
    E_mode, "the integrated normal-mode motion conserves energy", rel_drift=1e-6
)
validate.close(
    max_leakage,
    0.0,
    "a pure normal mode does not excite the other mode (no leakage)",
    atol=1e-6,
)

## Exercise 6 — A general motion is a superposition of modes (student animation)

A pure mode is special; a *general* small-amplitude motion is a **superposition**
of all the modes, each oscillating independently with an amplitude and phase set
by the initial conditions. Projecting the initial state onto the modes through
the mass matrix gives the modal coefficients $c_k = \mathbf v_k^{\mathsf T} M\,
\boldsymbol\eta_0$ (and $s_k = \mathbf v_k^{\mathsf T} M\,\dot{\boldsymbol\eta}_0$
for the initial velocities), after which {eq}`eq-mortho` guarantees the exact
closed-form evolution

$$ \boldsymbol\eta(t) = \sum_k \Big(c_k\cos\omega_k t
   + \tfrac{s_k}{\omega_k}\sin\omega_k t\Big)\,\mathbf v_k . $$

We release the double pendulum from a *generic* small displacement (only the
upper angle displaced) at rest, so both modes are excited.

1. Decompose the initial condition onto the modes,
   $c_k = \mathbf v_k^{\mathsf T} M\,\boldsymbol\eta_0$, and reconstruct
   $\boldsymbol\eta(t)$ from the sum above.
2. **Build the animation**: three panels side by side (the full motion and
   the two modal contributions $c_k\cos(\omega_k t)\,\mathbf v_k$), so you
   can watch the full motion *be* the sum of its modes. You have the angle
   time series below; assemble the `FuncAnimation`, `plt.close(fig)`, then
   `show(...)`.
3. Confirm the modal superposition reproduces a direct integration of
   {eq}`eq-modes-eom` (`scipy.integrate.solve_ivp`, `DOP853`).

> A ✗ on the check is about the **modal coefficients / reconstruction**, not the
> animation: any correct drawing of the same trajectories is fine. If it fails,
> inspect the $c_k = \mathbf v_k^{\mathsf T} M\,\boldsymbol\eta_0$ and the
> per-mode time evolution.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    eta_modal,
    eta_int,
    "the normal-mode superposition reproduces the directly integrated motion",
    rtol=1e-6,
    atol=1e-9,
)

## Exercise 7 — Stability from the spectrum (the contrast)

Everything so far assumed a stable equilibrium, where $K$ is positive definite
and every $\omega_k^2 > 0$. Now break that. The double pendulum has another
equilibrium with the **upper** bob hanging down but the **lower** bob balanced
straight up, $\boldsymbol\theta_0 = (0, \pi)$: a saddle of the potential. There
the stiffness matrix is *indefinite*: linearising gives $K = mg\ell\,
\operatorname{diag}(2, -1)$, with one negative eigenvalue. By the stability
argument above, that mode has $\omega^2 < 0$, so its $e^{i\omega t}$ becomes a
real exponential: the inverted bob topples rather than oscillates.

1. Linearise about $\boldsymbol\theta_0 = (0,\pi)$, solve the generalised
   eigenproblem {eq}`eq-geneig` with `scipy.linalg.eigh(K, M)`, and confirm
   one $\omega^2 < 0$.
2. Integrate {eq}`eq-modes-eom` with `scipy.integrate.solve_ivp` (`DOP853`)
   from a tiny displacement and plot the two normal coordinates: one
   oscillates, the other grows exponentially.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    eigvals_inv.min() < 0 and np.abs(xi_inv[0, -1]) > 50 * np.abs(xi_inv[0, 0]),
    "an unstable equilibrium has a negative ω² — a growing mode, not an oscillation",
    f"min ω² = {eigvals_inv.min():.3f}, |ξ| grew by {np.abs(xi_inv[0, -1] / xi_inv[0, 0]):.0f}×",
)

## Notebook summary

- The linearisation of a general Lagrangian into mass and stiffness matrices
  $M,K$ with the `linearize` helper built here (packaged for the series as
  `ecp.mechanics.linearize`), and the generalised eigenproblem
  $K\mathbf v=\omega^2 M\mathbf v$ solved with `scipy.linalg.eigh(K, M)`
  (packaged as `ecp.mechanics.normal_modes`).
- $M$-orthonormality of the modes ($V^\top M V=I$) and simultaneous diagonalisation; the
  in-phase and out-of-phase mode shapes; a pure mode oscillating at one frequency; a general
  motion as a superposition of modes; and stability read from the spectrum.

## Outlook

- **Molecular vibrations.** The longitudinal normal modes of a CO₂-like linear
  chain of three masses and two springs are a zero-frequency uniform translation
  and the symmetric and antisymmetric stretches; the translation reflects the
  chain's translational symmetry, tying back to the momentum conservation of
  [§2.2](noether.ipynb). The full three-dimensional molecule adds a
  doubly-degenerate bending mode, its degeneracy forced by symmetry.
- **The continuum limit.** Letting the spring chain of [§1.5](../01-elementary-mechanics/coupled-oscillators.ipynb) grow to
  $N\to\infty$ turns the discrete normal modes into **phonons** with a dispersion
  relation: the same generalised eigenproblem, now the harmonic theory of
  solids.
- **Damped and driven modes.** Adding dissipation and a periodic drive makes each
  normal coordinate a damped driven oscillator ([§1.2](../01-elementary-mechanics/damped-driven-pendulum.ipynb)), so a multi-DOF
  system shows a resonance peak at each $\omega_k$.
- **Forward links.** This same harmonic approximation (diagonalising the
  Hessian of a potential) is the basis of normal-mode analysis in molecular
  dynamics (Volume VI) and of field quantisation in terms of modes.

### References

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()